# High-rise 02 - Pressure coefficient (Cp)

Compute `Cp = (p - p_ref) / q`, with `q = 0.5 * rho * U_H^2` from the case (using
the `u_ref` measured in notebook 01). Writes the Cp time series to storage for
notebook 03 and a stats summary to `deliverables/`.

Defaults run on the `galpao` fixture (`u_h=0.05`, `rho=1.0` -> `q=0.00125`, matching
the `cp.yaml` template). For a real case, set `CFDMOD_HR_CASE_DATA` to a `case_data`
dir and `CFDMOD_HR_DATA_DIR` / `CFDMOD_HR_BODY_KEY` / `CFDMOD_HR_REF_KEY` to the data.

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

from cfdmod import high_rise as hr
from cfdmod.high_rise import plotting


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plotting.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
import json

from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

# --- config -------------------------------------------------------------
DATA_DIR = pathlib.Path(os.environ.get("CFDMOD_HR_DATA_DIR", FIX / "pressure" / "data"))
BODY_KEY = os.environ.get("CFDMOD_HR_BODY_KEY", "bodies.galpao")
REF_KEY = os.environ.get("CFDMOD_HR_REF_KEY", "points.static_pressure")
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)

# Build the case: from a real case_data dir, or an example tuned to the mesh.
CASE_DATA = os.environ.get("CFDMOD_HR_CASE_DATA")
PARAMS = os.environ.get("CFDMOD_HR_PARAMS", "params_cat3.yaml")
if CASE_DATA:
    case = hr.HighRiseCase.from_case_data(pathlib.Path(CASE_DATA), PARAMS)
else:
    case = hr.example_high_rise_case(MESH)

# Apply the reference velocity measured in notebook 01, if available.
ref_json = OUTPUT_BASE / "deliverables" / VERSION / "inflow" / "reference_velocity.json"
u_env = os.environ.get("CFDMOD_HR_UH")
if u_env:
    case = case.with_reference_velocity(float(u_env))
elif ref_json.exists():
    u = json.loads(ref_json.read_text()).get("u_ref")
    if u:
        case = case.with_reference_velocity(float(u))
print(f"U_H = {case.u_h:.5g} m/s -> dynamic pressure q = {case.dynamic_pressure:.5g} Pa")

In [ ]:
# --- compute Cp ---------------------------------------------------------
storage = XdmfH5Storage(DATA_DIR)
body = storage.read_data_source(pathlib.Path(BODY_KEY))
p_ref = storage.read_data_source(pathlib.Path(REF_KEY))
print(f"body: {body.kind}, {body.n_elements} elements, {body.time.n_timesteps} steps")

cp = hr.cp_from_pressure(body, p_ref, case)
cp_stats = hr.cp_from_pressure(body, p_ref, case, statistics=["mean", "rms", "min", "max"])
mean_cp = cp_stats.fields.read("mean")
print(f"Cp mean range: [{np.nanmin(mean_cp):.3f}, {np.nanmax(mean_cp):.3f}]")

In [ ]:
# --- write Cp time series to storage for notebook 03 --------------------
ARTIFACTS = OUTPUT_BASE / "artifacts" / VERSION
ARTIFACTS.mkdir(parents=True, exist_ok=True)
art_storage = XdmfH5Storage(ARTIFACTS)
art_storage.write_data_source(pathlib.Path("cp.time_series"), cp)
print("wrote", ARTIFACTS / "cp.time_series.h5")

In [ ]:
import pandas as pd

# --- Cp stats summary -> deliverables -----------------------------------
dbg = hr.DebugWriter(OUTPUT_BASE, stage="cp", version=VERSION)
fig, ax = plotting.new_axes(xlabel="mean Cp [-]", ylabel="count", title="Cp mean distribution")
ax.hist(mean_cp[np.isfinite(mean_cp)], bins=40)
dbg.savefig(fig, "cp_mean_hist.png")
plotting.close(fig)

stat_names = ["mean", "rms", "min", "max"]
summary = pd.DataFrame(
    {
        "stat": stat_names,
        "field_mean": [float(np.nanmean(cp_stats.fields.read(s))) for s in stat_names],
    }
)
summary.to_csv(dbg.deliverable_path("cp_summary.csv"), index=False)
summary